In [ ]:
# Root directory containing TITO's `results/` output tree (as written by
# `scripts/sample.py` + `scripts/analyse.py`, run from the repository root).
RESULTS_DIR = "../results"


# Paper figures


In [ ]:
import os.path as osp
import os
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.font_manager as fm
from matplotlib import gridspec

font_size = 14
font = {#'family': 'Liberation Sans',
    'weight' : 'normal',
    'size'   : font_size}


#matplotlib.rc('font', **font)

font_path = None
font_prop = fm.FontProperties(fname=font_path, size=font["size"], weight='normal')

palette_colors = ["#830051", "#68D2DF", "#F0AB00", "#83928E", "#cad502", "#e6007b","#515151"] #purple, blue, orange, grey, green, pink

In [ ]:
import glob
from scipy.spatial.distance import jensenshannon


def aggregate_model_vs_md_metrics(data_set, sub_data_set, model, split, lag, nested_samples,
                                   ode_steps=None, top_n=10, bins=100, results_dir=RESULTS_DIR):
    """
    Reproduces the test-set-wide `tica_js`, `vamp_gaps`, `rel_errors` arrays used
    in Figure 2 directly from the per-molecule files `scripts/analyse.py` writes,
    since the original aggregate `.npy` files are not produced by any script in
    this repository. Aggregates over every test molecule that has a result for
    the given (lag, nested_samples, ode_steps).
    """
    suffix = f"_ode_steps_{ode_steps}" if ode_steps is not None else ""
    md_dir = f"{results_dir}/{data_set}/{sub_data_set}/md/{split}"
    model_dir = f"{results_dir}/{data_set}/{sub_data_set}/{model}/{split}"

    tica_js, vamp_gaps, rel_errors = [], [], []
    for mol_dir in sorted(glob.glob(f"{md_dir}/mol_*")):
        mol_tag = osp.basename(mol_dir)
        model_mol_dir = f"{model_dir}/{mol_tag}"
        proj_path = f"{model_mol_dir}/tica_projections_random_init_lag_{int(lag)}_nested_{nested_samples}{suffix}.npy"
        gap_path = f"{model_mol_dir}/vamp_gap_random_init_lag_{int(lag)}_nested_{nested_samples}{suffix}.npy"
        svs_path = f"{model_mol_dir}/vamp_singular_values_random_init_lag_{int(lag)}_nested_{nested_samples}{suffix}.npy"
        if not (osp.exists(proj_path) and osp.exists(gap_path) and osp.exists(svs_path)):
            continue  # molecule not sampled with this configuration

        md_projections = np.load(f"{mol_dir}/tica_projections.npy").reshape(-1, 2)
        model_projections = np.load(proj_path).reshape(-1, 2)

        np.random.seed(42)
        n = min(len(md_projections), len(model_projections))
        md_sub = md_projections[np.random.choice(len(md_projections), size=n, replace=False)]
        model_sub = model_projections[np.random.choice(len(model_projections), size=n, replace=False)]

        counts_md, xedges, yedges = np.histogram2d(md_sub[:, 0], md_sub[:, 1], bins=bins)
        counts_model, _, _ = np.histogram2d(model_sub[:, 0], model_sub[:, 1], bins=[xedges, yedges])
        tica_js.append(jensenshannon(counts_md.flatten(), counts_model.flatten()) ** 2)

        vamp_gaps.append(float(np.load(gap_path)))

        md_svs = np.load(f"{mol_dir}/vamp_singular_values.npy")
        model_svs = np.load(svs_path)
        md_ts = -lag / np.log(md_svs)
        model_ts = -lag / np.log(model_svs)
        rel_errors.append(np.mean(np.abs((model_ts[:top_n] - md_ts[:top_n]) / md_ts[:top_n])))

    return np.array(tica_js), np.array(vamp_gaps), np.array(rel_errors)


## Figure 2: Global metrics + example

In [ ]:
font_size = 12
font = {#'family': 'Liberation Sans',
    'weight' : 'normal',
    'size'   : font_size}


#matplotlib.rc('font', **font)

font_path = None
font_prop = fm.FontProperties(fname=font_path, size=font["size"], weight='normal')

In [ ]:
# Load examples data
top_n = 10
lags = [57, 250] #mdqm9, timewarp

md = []
edges = []
tito = []
md_timescales = []
tito_timescales = []
vamp2_gap = []

tica_js = []
vamp_gaps = []
rel_errors = []

# mdqm9
data_set = "mdqm9"
model = "nfksixdq" # uniform: gn10le53, nfksixdq . exp: g2myka6s, wuck6wrs
split = "test"
mol_index = 19 #state discovery candidates, 12,13, 21, 25, 29
lag = lags[0]
nested_samples = 640

mol_tica_js, mol_vamp_gaps, mol_rel_errors = aggregate_model_vs_md_metrics(
    data_set, "version_0", model, split, lag, nested_samples, top_n=top_n)
tica_js.append(mol_tica_js)
vamp_gaps.append(mol_vamp_gaps)
rel_errors.append(mol_rel_errors)

md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}.npy")
md_projections = md_projections.reshape(-1, md_projections.shape[-1])
model_projections = model_projections.reshape(-1, model_projections.shape[-1])

# Subsample model projections to match MD projections size
np.random.seed(42)  # Set seed for reproducibility
indices = np.random.choice(len(model_projections), size=len(md_projections), replace=False)
model_projections_subsampled = model_projections[indices]

# Calculate histograms from re projections
counts_md, xedges, yedges = np.histogram2d(model_projections_subsampled[:,0], model_projections_subsampled[:,1], bins=100)
counts_md = counts_md.T  # Transpose to match the orientation of pcolormesh
counts_model, _, _ = np.histogram2d(model_projections[:,0], model_projections[:,1], bins=[xedges, yedges])
counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh

# Normalize histograms
pmf_md = counts_md / np.sum(counts_md)
pmf_model = counts_model / np.sum(counts_model)

# Calculate negative log of probability (free energy)
with np.errstate(divide='ignore'):
    fe_md = -np.log(pmf_md)
    fe_model = -np.log(pmf_model)

# Replace infinities with NaN
fe_md[~np.isfinite(fe_md)] = np.nan
fe_model[~np.isfinite(fe_model)] = np.nan

md_svs = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values.npy")
model_svs = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values_random_init_lag_{int(lag)}_nested_{nested_samples}.npy")
vamp2_gap.append(np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_gap_random_init_lag_{int(lag)}_nested_{nested_samples}.npy"))
#compute implied timescales
md_timescales.append(-lag/np.log(md_svs))
tito_timescales.append(-lag/np.log(model_svs))

md.append(fe_md)
tito.append(fe_model)
edges.append((xedges, yedges))

# timewarp

data_set = "timewarp"
sub_data_set = "huge"  # "large", "huge"
model = "lxf356pw" # uniform: 57hk8qzx, lxf356pw. exp: ktgl41l5, dg5fl7wv
split = "test"
mol_index = 84 #1 , 84
lag = lags[1]  #250 125
nested_samples = 500  #500 400
ode_steps = 40

mol_tica_js, mol_vamp_gaps, mol_rel_errors = aggregate_model_vs_md_metrics(
    data_set, sub_data_set, model, split, lag, nested_samples, ode_steps=ode_steps, top_n=top_n)
tica_js.append(mol_tica_js)
vamp_gaps.append(mol_vamp_gaps)
rel_errors.append(mol_rel_errors)

md_projections = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
model_projections = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")#_ode_steps_{ode_steps} short_params/

md_projections = md_projections.reshape(-1, md_projections.shape[-1])
model_projections = model_projections.reshape(-1, model_projections.shape[-1])

# Subsample model projections to match MD projections size
np.random.seed(42)  # Set seed for reproducibility
indices = np.random.choice(len(md_projections), size=len(model_projections), replace=False)
md_projections_subsampled = md_projections[indices]

# Calculate histograms from re projections
counts_md, xedges, yedges = np.histogram2d(md_projections_subsampled[:,0], md_projections_subsampled[:,1], bins=100)
counts_md = counts_md.T  #  Transpose to match the orientation of pcolormesh
counts_model, _, _ = np.histogram2d(model_projections[:,0], model_projections[:,1], bins=[xedges, yedges])
counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh       

# Normalize histograms
pmf_md = counts_md / np.sum(counts_md)
pmf_model = counts_model / np.sum(counts_model) 

# Calculate negative log of probability (free energy)
with np.errstate(divide='ignore'):
    fe_md = -np.log(pmf_md)
    fe_model = -np.log(pmf_model)

# Replace infinities with NaN
fe_md[~np.isfinite(fe_md)] = np.nan
fe_model[~np.isfinite(fe_model)] = np.nan

md_svs = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/md/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values.npy")
model_svs = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values_random_init_lag_{int(lag)}_nested_{nested_samples}_ode_steps_{ode_steps}.npy") #short_params. _ode_steps_{ode_steps}
vamp2_gap.append(np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_gap_random_init_lag_{int(lag)}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")) ##short_params. _ode_steps_{ode_steps}
#compute implied timescales
md_timescales.append(-lag/np.log(md_svs))
tito_timescales.append(-lag/np.log(model_svs))

md.append(fe_md)
tito.append(fe_model)
edges.append((xedges, yedges))

In [ ]:

bins_dsets = [50, 10]

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(13, 16))

# Outer grid: 2 blocks stacked vertically (can extend if more systems)
outer = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.25)

# -----------------------------
# Loop over blocks
# -----------------------------
for block_idx in range(2):  # two stacked blocks
    # Each block is a 2x3 grid
    inner = gridspec.GridSpecFromSubplotSpec(
        2, 3,
        subplot_spec=outer[block_idx],
        hspace=0.35,
        wspace=0.1,               # overall horizontal spacing
        width_ratios=[1, 1, 1.5]  # shrink spacing effect between col1 & col2
    )
    
    # --- Top row (2D density + scatter)
    ax1 = fig.add_subplot(inner[0, 0])  # MD density
    ax2 = fig.add_subplot(inner[0, 1])  # Model density
    ax3 = fig.add_subplot(inner[0, 2])  # Timescale scatter
    
    # --- Bottom row (histograms)
    ax4 = fig.add_subplot(inner[1, 0])  # TICA divergence
    ax5 = fig.add_subplot(inner[1, 1])  # VAMP-2 gap
    ax6 = fig.add_subplot(inner[1, 2])  # Relative timescale error
    
    # -----------------------------
    # Example placeholders (replace with real plots)
    # -----------------------------
    ax1.set_title("MD", fontproperties=font_prop)
    ax1.set_xlabel('TIC 1', fontproperties=font_prop)
    ax1.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    

    ax2.set_title("TITO", fontproperties=font_prop)
    ax2.set_xlabel('TIC 1', fontproperties=font_prop)
    ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax2.get_position().x1 + 0.01, 
                            ax2.get_position().y0, 
                            0.01, 
                            ax2.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    labelpad = 5 if block_idx == 0 else 1
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=labelpad)

    # Plot timescale scatter
    ax3.scatter(md_timescales[block_idx], tito_timescales[block_idx], color=palette_colors[1], s=100, zorder=2)
    rel_error = np.mean(np.abs((tito_timescales[block_idx][:10] - md_timescales[block_idx][:10]) / md_timescales[block_idx][:10]))

    #Set log scale
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_box_aspect(1)

    #Set equal range in both axis
    # Get current limits
    xmin, xmax = ax3.get_xlim()
    ymin, ymax = ax3.get_ylim()

    # Find the combined min and max
    low = min(xmin, ymin)
    high = max(xmax, ymax)

    ax3.plot(np.arange(low, high, (high-low)/10), np.arange(low, high, (high-low)/10), color=palette_colors[0], linestyle='dashed',
        linewidth=2, markersize=12, zorder=1, label='y=x')

    # Apply to both ax3
    ax3.set_xlim(low, high)
    ax3.set_ylim(low, high)

    ax3.plot([lags[block_idx] for _ in range(10)], np.arange(low, lags[block_idx], (lags[block_idx]-low)/10), color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)
    ax3.plot(np.arange(low, lags[block_idx], (lags[block_idx]-low)/10), [lags[block_idx] for _ in range(10)], color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)

    ax3.text(0.05, 0.9, f'Relative error = {rel_error:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    ax3.text(0.05, 0.8, f'VAMP-2 gap  = {vamp2_gap[block_idx]:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    plt.tick_params(axis='both', which='major', labelsize=font["size"])

    ax3.set_xlabel('Time-scale MD [$ps$]', fontproperties=font_prop, labelpad=1)
    ax3.set_ylabel('Time-scale TITO [$ps$]', fontproperties=font_prop, labelpad=1)

    # Set the same aspect ratio for histogram plots
    ax4.hist(tica_js[block_idx], bins=bins_dsets[block_idx], color=palette_colors[1], edgecolor='black')
    ax4.text(0.45, 0.9, f'Mean = {np.mean(tica_js[block_idx]):.3f}', transform=ax4.transAxes, fontproperties=font_prop)
    ax4.text(0.45, 0.8, f'Median  = {np.median(tica_js[block_idx]):.3f}', transform=ax4.transAxes, fontproperties=font_prop)  
    ax4.set_xlabel('Jensen-Shannon divergence', fontproperties=font_prop)
    ax4.set_ylabel('Count', fontproperties=font_prop)
    ax4.set_box_aspect(1)  # Make it square
    #ax4.annotate('', xy=(0.1, 0.1), xytext=(0.12, 0.12),
    #             arrowprops=dict(facecolor='black', lw=2))

    ax5.hist(vamp_gaps[block_idx], bins=bins_dsets[block_idx], color=palette_colors[1], edgecolor='black')
    ax5.text(0.05, 0.9, f'Mean = {np.mean(vamp_gaps[block_idx]):.3f}', transform=ax5.transAxes, fontproperties=font_prop)
    ax5.text(0.05, 0.8, f'Median  = {np.median(vamp_gaps[block_idx]):.3f}', transform=ax5.transAxes, fontproperties=font_prop)
    ax5.set_xlabel('VAMP-2 gap', fontproperties=font_prop)
    ax5.set_ylabel('Count', fontproperties=font_prop)
    ax5.set_box_aspect(1)  # Make it square
    
    ax6.hist(rel_errors[block_idx], bins=bins_dsets[block_idx], color=palette_colors[1], edgecolor='black')
    ax6.text(0.45, 0.9, f'Mean = {np.mean(rel_errors[block_idx]):.3f}', transform=ax6.transAxes, fontproperties=font_prop)
    ax6.text(0.45, 0.8, f'Median  = {np.median(rel_errors[block_idx]):.3f}', transform=ax6.transAxes, fontproperties=font_prop)
    ax6.set_xlabel('Relative time-scale discrepancy', fontproperties=font_prop)
    ax6.set_ylabel('Count', fontproperties=font_prop)
    ax6.set_box_aspect(1)  # Make it square

    # Add titles for blocks
    if block_idx == 0:
        fig.text(0.5, 0.91, "Small Molecules", fontsize=16, ha='center', fontproperties=font_prop)
    else:
        fig.text(0.5, 0.48, "Tetra-peptides", fontsize=16, ha='center', fontproperties=font_prop)

    # Set common font properties for all axes in this block
    for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
        for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
                    ax.get_xticklabels() + ax.get_yticklabels()):
            item.set_fontproperties(font_prop)

# Adjust layout to accommodate all elements
plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space on the right for colorbars
plt.show()


In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/main/fig3_global.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/main/fig3_global.pdf", transparent=True)

In [ ]:
#TODO after base: add arrows and molecules

## Figure 3: Equilibrium distribution analysis


In [ ]:
from scipy.spatial.distance import jensenshannon


def compute_overlap_based_metrics(md_projections, model_projections, mdft_projections, re_projections, num_bins=100):
    # Combine the data to find a common range for histogram bins
    all_data = np.concatenate((md_projections, model_projections, mdft_projections, re_projections), axis=0)

    # Determine bin edges
    x_bins = np.linspace(all_data[:, 0].min(), all_data[:, 0].max(), num_bins + 1)
    y_bins = np.linspace(all_data[:, 1].min(), all_data[:, 1].max(), num_bins + 1)

    # Compute 2D histograms for each array.
    # The `histogram2d` function returns a 2D array of counts.
    hist1, _, _ = np.histogram2d(md_projections[:, 0], md_projections[:, 1], bins=[x_bins, y_bins])
    hist2, _, _ = np.histogram2d(model_projections[:, 0], model_projections[:, 1], bins=[x_bins, y_bins])
    hist3, _, _ = np.histogram2d(mdft_projections[:, 0], mdft_projections[:, 1], bins=[x_bins, y_bins])
    hist4, _, _ = np.histogram2d(re_projections[:, 0], re_projections[:, 1], bins=[x_bins, y_bins])
    

    # Flatten the 2D histograms to 1D arrays
    hist1_flat = hist1.flatten()
    hist2_flat = hist2.flatten()
    hist3_flat = hist3.flatten()
    hist4_flat = hist4.flatten()

    # Normalize the histograms to get probability distributions (PMFs)
    pmf1 = hist1_flat / np.sum(hist1_flat)
    pmf2 = hist2_flat / np.sum(hist2_flat)
    pmf3 = hist3_flat / np.sum(hist3_flat)
    pmf4 = hist4_flat / np.sum(hist4_flat)

    nonzero_1 = pmf1 > 0
    nonzero_2 = pmf2 > 0
    nonzero_3 = pmf3 > 0
    nonzero_4 = pmf4 > 0

    nonzero_md_re = nonzero_1 & nonzero_4
    nonzero_model_re = nonzero_2 & nonzero_4
    nonzero_mdft_re = nonzero_3 & nonzero_4

    # Compute the Jensen-Shannon divergence
    js = [jensenshannon(pmf1, pmf4) ** 2, jensenshannon(pmf2, pmf4) ** 2, jensenshannon(pmf3, pmf4) ** 2]
    
    # Coverage
    cov = [np.sum(pmf4[nonzero_md_re]), np.sum(pmf4[nonzero_model_re]), np.sum(pmf4[nonzero_mdft_re])]

    # Precision
    precision = [np.sum(pmf1[nonzero_md_re]), np.sum(pmf2[nonzero_model_re]), np.sum(pmf3[nonzero_mdft_re])]

    return js, cov, precision

In [ ]:
#load violin plot data

data_set = "mdqm9"
model = "nfksixdq"
split = "test"
lag = 1000
nested_samples = 1000
ode_steps = 40
ft_ps = 10

indices =  np.arange(100)
js_md = []
js_model = []
js_mdft = []
cov_md = []
cov_model = []
cov_mdft = []
precision_md = []
precision_model = []
precision_mdft = []


for mol_index in indices:
    md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
    model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")
    md_projections = md_projections.reshape(-1, md_projections.shape[-1])
    model_projections = model_projections.reshape(-1, model_projections.shape[-1])

    re_projections_path = f"{RESULTS_DIR}/{data_set}/version_0/re/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_re.npy"
    re_projections = np.load(re_projections_path)
    re_projections = re_projections.reshape(-1, re_projections.shape[-1])

    mdft_projections_path = f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_ft_positions_{ft_ps}_ps_random_init_lag_{lag}_nested_{nested_samples}_ode_steps_{ode_steps}.npy"
    mdft_projections = np.load(mdft_projections_path)
    mdft_projections = mdft_projections.reshape(-1, mdft_projections.shape[-1])

    # Subsample to the same number of points for fair comparison
    min_data_points = min(re_projections.shape[0], model_projections.shape[0], md_projections.shape[0], mdft_projections.shape[0])
    model_projections = model_projections[np.random.choice(model_projections.shape[0], size=min_data_points, replace=False)]
    md_projections = md_projections[np.random.choice(md_projections.shape[0], size=min_data_points, replace=False)]
    re_projections = re_projections[np.random.choice(re_projections.shape[0], size=min_data_points, replace=False)]
    mdft_projections = mdft_projections[np.random.choice(mdft_projections.shape[0], size=min_data_points, replace=False)]

    js, cov_fee, precision = compute_overlap_based_metrics(md_projections, model_projections, mdft_projections, re_projections, num_bins=50)
    md_js, model_js, mdft_js = js
    md_cov, model_cov, mdft_cov = cov_fee
    md_precision, model_precision, mdft_precision = precision

    js_md.append(md_js)
    js_model.append(model_js)
    js_mdft.append(mdft_js)
    cov_md.append(md_cov)
    cov_model.append(model_cov)
    cov_mdft.append(mdft_cov)
    precision_md.append(md_precision)
    precision_model.append(model_precision)
    precision_mdft.append(mdft_precision)

js_md = np.array(js_md)
js_model = np.array(js_model)
js_mdft = np.array(js_mdft)
cov_md = np.array(cov_md)
cov_model = np.array(cov_model)
cov_mdft = np.array(cov_mdft)
precision_md = np.array(precision_md)
precision_model = np.array(precision_model)
precision_mdft = np.array(precision_mdft)

In [ ]:
# Load mol 21 data

data_set = "mdqm9"
model = "nfksixdq" # uniform: gn10le53, nfksixdq . exp: g2myka6s, wuck6wrs
split = "test"
mol_index =21 #state discovery candidates, 12,13,18, 21, 25, 29
lag = 1000
nested_samples = 50000

md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
md_projections_long = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_trained_short_predicted_long.npy")
model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_init_009994_lag_{lag}_nested_{nested_samples}_ode_steps_20.npy")
md_projections = md_projections.reshape(-1, md_projections.shape[-1])
#model_projections = model_projections.reshape(-1, model_projections.shape[-1])

model_projections = model_projections[:,10000:,:]
model_projections = model_projections.reshape(-1, model_projections.shape[-1])

md_projections_long = md_projections_long[:, 495:]
md_projections_long = md_projections_long.reshape(-1, md_projections_long.shape[-1])

re_projections_path = f"{RESULTS_DIR}/{data_set}/version_0/re/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_re.npy"
re_projections = np.load(re_projections_path)
re_projections = re_projections.reshape(-1, re_projections.shape[-1])

# Subsample to the same number of points for fair comparison
min_data_points = 100000#min(re_projections.shape[0], model_projections.shape[0], md_projections.shape[0])
model_projections = model_projections[np.random.choice(model_projections.shape[0], size=min_data_points, replace=False)]
md_projections_long = md_projections_long[np.random.choice(md_projections_long.shape[0], size=min_data_points, replace=False)]

#md_projections = md_projections[np.random.choice(md_projections.shape[0], size=min_data_points, replace=False)]
#re_projections = re_projections[np.random.choice(re_projections.shape[0], size=min_data_points, replace=False)]


# Calculate histograms from re projections
counts_re, xedges, yedges = np.histogram2d(re_projections[:,0], re_projections[:,1], bins=100)
counts_re = counts_re.T  # Transpose to match the orientation of pcolormesh
counts_md = np.histogram2d(md_projections[:,0], md_projections[:,1], bins=[xedges, yedges])[0]
counts_md = counts_md.T  # Transpose to match the orientation of pcolormesh
counts_model = np.histogram2d(model_projections[:,0], model_projections[:,1], bins=[xedges, yedges])[0]
counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh
counts_md_long = np.histogram2d(md_projections_long[:,0], md_projections_long[:,1], bins=[xedges, yedges])[0]
counts_md_long = counts_md_long.T  # Transpose to match the orientation of pcolormesh

# Normalize histograms
pmf_re = counts_re / np.sum(counts_re)
pmf_md = counts_md / np.sum(counts_md)
pmf_model = counts_model / np.sum(counts_model)
pmf_md_long = counts_md_long / np.sum(counts_md_long)

# Calculate negative log of probability (free energy)
with np.errstate(divide='ignore'):
    fe_re = -np.log(pmf_re)
    fe_md = -np.log(pmf_md)
    fe_model = -np.log(pmf_model)
    fe_md_long = -np.log(pmf_md_long)

# Replace infinities with NaN
fe_re[~np.isfinite(fe_re)] = np.nan
fe_md[~np.isfinite(fe_md)] = np.nan
fe_model[~np.isfinite(fe_model)] = np.nan
fe_md_long[~np.isfinite(fe_md_long)] = np.nan



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import gridspec

# -----------------------------
# Example data (replace with yours)
# -----------------------------
np.random.seed(42)


js_data = [js_md, js_model, js_mdft]
cov_data = [cov_md, cov_model, cov_mdft]
prec_data = [precision_md, precision_model, precision_mdft]

# -----------------------------
# Layout with gridspec
# -----------------------------
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 5, height_ratios=[1, 1, 1], width_ratios=[1, 1, 1, 1, .05], hspace=0.25, wspace=0.5)


colors = ["#9e3d64", "#3db7cc", "#e3b62b"]

# -----------------------------
# Top row: Single violin plot with 3 categories: JS divergence
# -----------------------------



# -----------------------------
# Mid top row: Single violin plot with 3 categories: Free energy excess
# -----------------------------
ax_mid_top = fig.add_subplot(gs[0, :])

# Prepare data
positions = [1, 2, 3]
labels = ['MD', 'TITO', 'TITO+MD']

# Create violin plot
violin_parts = ax_mid_top.violinplot(cov_data, positions=positions, showmeans=True, showmedians=True, showextrema=False)

# Customize colors
for i, pc in enumerate(violin_parts['bodies']):
    pc.set_facecolor(palette_colors[i])
    pc.set_alpha(0.5)

# Set color for mean and median lines
violin_parts['cmeans'].set_color('black')
violin_parts['cmedians'].set_color('white')
violin_parts['cmedians'].set_linewidth(1.5)

# Add scatter points for better visibility
for i, d in enumerate(cov_data):
    # Add jittered scatter points
    x = np.random.normal(positions[i], 0.05, size=len(d))
    ax_mid_top.scatter(x, d, s=5, color=palette_colors[i], alpha=0.3)

# Add statistics

for i, d in enumerate(cov_data):
    
    mean_val = np.mean(d)
    median_val = np.median(d)
    error = np.std(d) / np.sqrt(len(d)) * 1.96  # 95% CI

    ax_mid_top.text(positions[i], ax_mid_top.get_ylim()[0] + 0.2,
                f"Mean: {mean_val:.2f} ± {error:.2f}\nMedian: {median_val:.2f}",
                ha="center", va="top", fontproperties=font_prop)
    
    

# Customize plot
ax_mid_top.set_xticks(positions)
ax_mid_top.set_xticklabels(labels, fontproperties=font_prop)
ax_mid_top.set_ylabel('Coverage', fontproperties=font_prop)
ax_mid_top.tick_params(axis='both', which='major', labelsize=font["size"])

# Add a horizontal line at y=1 for reference
ax_mid_top.axhline(y=1, color='gray', linestyle='--', alpha=0.7)

# -----------------------------
# Mid row: Single violin plot with 3 categories: Free energy excess
# -----------------------------
ax_mid_bottom = fig.add_subplot(gs[1, :])

# Prepare data
positions = [1, 2, 3]
labels = ['MD', 'TITO', 'TITO+MD']

# Create violin plot
violin_parts = ax_mid_bottom.violinplot(prec_data, positions=positions, showmeans=True, showmedians=True, showextrema=False)

# Customize colors
for i, pc in enumerate(violin_parts['bodies']):
    pc.set_facecolor(palette_colors[i])
    pc.set_alpha(0.5)

# Set color for mean and median lines
violin_parts['cmeans'].set_color('black')
violin_parts['cmedians'].set_color('white')
violin_parts['cmedians'].set_linewidth(1.5)

# Add scatter points for better visibility
for i, d in enumerate(prec_data):
    # Add jittered scatter points
    x = np.random.normal(positions[i], 0.05, size=len(d))
    ax_mid_bottom.scatter(x, d, s=5, color=palette_colors[i], alpha=0.3)

# Add statistics

for i, d in enumerate(prec_data):
    
    mean_val = np.mean(d)
    median_val = np.median(d)
    error = np.std(d) / np.sqrt(len(d)) * 1.96  # 95% CI

    ax_mid_bottom.text(positions[i], ax_mid_bottom.get_ylim()[0] + 0.1,
                f"Mean: {mean_val:.2f} ± {error:.2f}\nMedian: {median_val:.2f}",
                ha="center", va="top", fontproperties=font_prop)
    
    

# Customize plot
ax_mid_bottom.set_xticks(positions)
ax_mid_bottom.set_xticklabels(labels, fontproperties=font_prop)
ax_mid_bottom.set_ylabel('Precision', fontproperties=font_prop)
ax_mid_bottom.tick_params(axis='both', which='major', labelsize=font["size"])

# Add a horizontal line at y=1 for reference
ax_mid_bottom.axhline(y=1, color='gray', linestyle='--', alpha=0.7)

# -----------------------------
# Bottom row: 2D log-density scatter
# -----------------------------

mid_bottom_ax = fig.add_subplot(gs[2, 2])

# RE projections subplot
im = mid_bottom_ax.pcolormesh(
    xedges, yedges, 
    fe_re,  # Transpose to match coordinates
    cmap="viridis_r",   # Use reversed viridis for high density as bright
    shading='auto'      # For smoother visualization
)
vmin, vmax = im.get_clim()
#cbar = fig.colorbar(im, ax=mid_bottom_ax)
#cbar.set_label("Free energy", fontproperties=font_prop)
#h_re = mid_bottom_ax.hist2d(re_projections[:, 0], re_projections[:, 1], bins=100, norm=LogNorm(), density=True)
mid_bottom_ax.set_title('RE', fontproperties=font_prop)
#mid_bottom_ax.set_xlabel('TIC 1', fontproperties=font_prop)
#mid_bottom_ax.set_ylabel('TIC 2', fontproperties=font_prop)
mid_bottom_ax.tick_params(axis='both', which='major', labelsize=font["size"])
mid_bottom_ax.set_aspect('equal')

left_bottom_ax = fig.add_subplot(gs[2, 0])

# MD projections subplot - switch to pcolormesh like RE
im_md = left_bottom_ax.pcolormesh(
    xedges, yedges, 
    fe_md,  # Using the free energy landscape like RE subplot
    cmap="viridis_r",
    shading='auto',
    vmin=vmin, vmax=vmax  # Ensure same color scale as RE subplot
)

#cbar = fig.colorbar(im_md, ax=left_bottom_ax)
#cbar.set_label("Free energy", fontproperties=font_prop)
left_bottom_ax.set_title('Short MD', fontproperties=font_prop)
#left_bottom_ax.set_xlabel('TIC 1', fontproperties=font_prop)
left_bottom_ax.set_ylabel('TIC 2', fontproperties=font_prop)
left_bottom_ax.tick_params(axis='both', which='major', labelsize=font["size"])
left_bottom_ax.set_aspect('equal')

left_mid_bottom_ax = fig.add_subplot(gs[2, 1])

# MD projections subplot - switch to pcolormesh like RE
im_md = left_mid_bottom_ax.pcolormesh(
    xedges, yedges, 
    fe_md_long,  # Using the free energy landscape like RE subplot
    cmap="viridis_r",
    shading='auto',
    vmin=vmin, vmax=vmax  # Ensure same color scale as RE subplot
)

#cbar = fig.colorbar(im_md, ax=left_mid_bottom_ax)
#cbar.set_label("Free energy", fontproperties=font_prop)
left_mid_bottom_ax.set_title('Long MD', fontproperties=font_prop)
#left_bottom_ax.set_xlabel('TIC 1', fontproperties=font_prop)
#left_mid_bottom_ax.set_ylabel('TIC 2', fontproperties=font_prop)
left_mid_bottom_ax.tick_params(axis='both', which='major', labelsize=font["size"])
left_mid_bottom_ax.set_aspect('equal')

right_bottom_ax = fig.add_subplot(gs[2, 3])

# Model projections subplot - switch to pcolormesh like RE
im_model = right_bottom_ax.pcolormesh(
    xedges, yedges, 
    fe_model,  # Using the free energy landscape like RE subplot
    cmap="viridis_r",
    shading='auto',
    vmin=vmin, vmax=vmax  # Ensure same color scale as RE subplot
)

right_bottom_ax.set_title('TITO', fontproperties=font_prop)
#right_bottom_ax.set_xlabel('TIC 1', fontproperties=font_prop)
#right_bottom_ax.set_ylabel('TIC 2', fontproperties=font_prop)
right_bottom_ax.tick_params(axis='both', which='major', labelsize=font["size"])
right_bottom_ax.set_aspect('equal')

# Create a separate axes for the colorbar to avoid affecting the main plot size
cbar_ax = fig.add_axes([right_bottom_ax.get_position().x1 + 0.01, 
                        right_bottom_ax.get_position().y0, 
                        0.01, 
                        right_bottom_ax.get_position().height])
cbar = fig.colorbar(im_model, cax=cbar_ax)
cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
# Apply font properties to tick labels
cbar.ax.tick_params(labelsize=font_prop.get_size())


# Add labels to rows
# Create labels A, B, C for the three rows

ax_mid_top.text(-0.075, 1.05, 'A', transform=ax_mid_top.transAxes, 
            fontsize=16, fontweight='bold', va='top', ha='right')

ax_mid_bottom.text(-0.075, 1.05, 'B', transform=ax_mid_bottom.transAxes, 
            fontsize=16, fontweight='bold', va='top', ha='right')

left_bottom_ax.text(-0.435, 1.05, 'C', transform=left_bottom_ax.transAxes, 
                    fontsize=16, fontweight='bold', va='top', ha='right')

# Add a common x-label for TIC 1 at the bottom centered under all plots
fig.text(0.48, 0.075, 'TIC 1', ha='center', fontproperties=font_prop)
plt.tight_layout()
plt.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/main/fig3_long_term.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/main/fig3_long_term.pdf", transparent=True)

In [ ]:
js_data = [js_md, js_model, js_mdft]

# -----------------------------
# Layout with gridspec
# -----------------------------
fig, ax_top = plt.subplots(figsize=(16, 4))

# Prepare data
positions = [1, 2, 3]
labels = ['MD', 'TITO', 'TITO+MD']

# Create violin plot
violin_parts = ax_top.violinplot(js_data, positions=positions, showmeans=True, showmedians=True, showextrema=False)

# Customize colors
for i, pc in enumerate(violin_parts['bodies']):
    pc.set_facecolor(palette_colors[i])
    pc.set_alpha(0.5)

# Set color for mean and median lines
violin_parts['cmeans'].set_color('black')
violin_parts['cmedians'].set_color('white')
violin_parts['cmedians'].set_linewidth(1.5)

# Add scatter points for better visibility
for i, d in enumerate(js_data):
    # Add jittered scatter points
    x = np.random.normal(positions[i], 0.05, size=len(d))
    ax_top.scatter(x, d, s=5, color=palette_colors[i], alpha=0.3)

# Add statistics

for i, d in enumerate(js_data):
    
    mean_val = np.mean(d)
    median_val = np.median(d)
    error = np.std(d) / np.sqrt(len(d)) * 1.96  # 95% CI

    ax_top.text(positions[i], ax_top.get_ylim()[1] * 0.95,
                f"Mean: {mean_val:.2f} ± {error:.2f}  \nMedian: {median_val:.2f}",
                ha="center", va="top", fontproperties=font_prop)
    
    

# Customize plot
ax_top.set_xticks(positions)
ax_top.set_xticklabels(labels, fontproperties=font_prop)
ax_top.set_ylabel('RE JS-divergence', fontproperties=font_prop)
ax_top.tick_params(axis='both', which='major', labelsize=font["size"])

# Add a horizontal line at y=0 for reference
ax_top.axhline(y=0, color='gray', linestyle='--', alpha=0.7)

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/appendix/js_long_term.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/appendix/js_long_term.pdf", transparent=True)

## Figure 4: Marginals


In [ ]:
import os.path as osp
import os
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.font_manager as fm

font_size = 14
font = {#'family': 'Liberation Sans',
    'weight' : 'normal',
    'size'   : font_size}


#matplotlib.rc('font', **font)

font_path = None
font_prop = fm.FontProperties(fname=font_path, size=font["size"], weight='normal')

palette_colors = ["#830051", "#68D2DF", "#F0AB00", "#83928E", "#cad502", "#e6007b","#515151"] #purple, blue, orange, grey, green, pink


In [ ]:
from deeptime.clustering import BoxDiscretization
from deeptime.markov import TransitionCountEstimator
def build_msm_discretization(ticsv, lag_time=1, n_bins=100):
    discretization = BoxDiscretization(
        2, n_bins
    ).fit_fetch(ticsv) #, v0=[np.min(phipsi[:, 0]), np.max(phipsi[:, 0])], v1=[np.min(phipsi[:, 1]), np.max(phipsi[:, 1])]

    dtrajs = discretization.transform(ticsv)
    count_model = TransitionCountEstimator(lag_time, count_mode="sliding", n_states=n_bins**2, sparse=False).fit_fetch(dtrajs)
    TPM = count_model.count_matrix / (
        count_model.count_matrix.sum(axis=1)[:, None] + 1e-8
    )
    return TPM, discretization

def conditional_probability(markov_model, discretization, x, y):
    x_state = discretization.transform(np.array([[x[0], y[0]]]))[0]
    conditional_prob = markov_model[x_state, :]
    conditional_prob /= np.sum(conditional_prob)
    return conditional_prob

class MSM:
    def __init__(self, n_bins, tic1, tic2):
        self.n_bins = n_bins
        self.true_tic1, self.true_tic2 = tic1, tic2
    def get_conditional_prob(self, tic1, tic2, lag):
        markov_model, discretization = build_msm_discretization(
            np.vstack([self.true_tic1, self.true_tic2]).T, lag, self.n_bins
        )
        cond_prob = conditional_probability(
            markov_model, discretization, tic1, tic2
        ).reshape(self.n_bins, self.n_bins)
        marginal_tic1 = cond_prob.sum(axis=0)
        marginal_tic2 = cond_prob.sum(axis=1)

        return cond_prob, marginal_tic1, marginal_tic2

    def __call__(self, tic1, tic2, lag):
        return self.get_conditional_prob(tic1, tic2, lag)


In [ ]:
import os

#load data for marginal TICAs

data_set = "timewarp"
sub_data_set = "huge"  # "large", "huge"
model = "ktgl41l5" # uniform: 57hk8qzx, lxf356pw. exp: ktgl41l5, dg5fl7wv
split = "test"
mol_index = 9
ck_nested_steps = 5
#ode_steps = 20

md_projections = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
tic1=md_projections[0,:,0]
tic2=md_projections[0,:,1]
initial_ticas = [tic1[0], tic2[0]]
msm = MSM(32, tic1, tic2)

lags = [10, 20, 100, 200, 1000] 
initial_conf = 0

# List to store MSM and model data for each lag
msm_data = []
model_data = []

for i_lag in range(len(lags)):
    lag = lags[i_lag]
    # MSM
    cond_pdf, mtic1, mtic2 = msm(np.array([tic1[initial_conf] for _ in range(1000000)]), np.array([tic2[initial_conf] for _ in range(1000000)]), lag)
    
    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        neg_log_cond_pdf = -np.log(cond_pdf)
    neg_log_cond_pdf[~np.isfinite(neg_log_cond_pdf)] = np.nan  # Replace infinities with NaN
    
    marginal_tic1_msm = np.sum(cond_pdf, axis=0)
    marginal_tic2_msm = np.sum(cond_pdf, axis=1)
    bin_edges_tic1 = np.linspace(np.min(tic1), np.max(tic1), len(marginal_tic1_msm) + 1)
    bin_widths_tic1 = np.diff(bin_edges_tic1)
    marginal_tic1_msm /= marginal_tic1_msm.mean() * bin_widths_tic1.sum()
    marginal_tic1_msm = np.append(marginal_tic1_msm, marginal_tic1_msm[-1])
    
    # Calculate negative log of marginal probabilities
    with np.errstate(divide='ignore'):
        neg_log_marginal_tic1 = -np.log(marginal_tic1_msm)
    neg_log_marginal_tic1[~np.isfinite(neg_log_marginal_tic1)] = np.nan
    
    bin_edges_tic2 = np.linspace(np.min(tic2), np.max(tic2), len(marginal_tic2_msm) + 1)
    bin_widths_tic2 = np.diff(bin_edges_tic2)
    marginal_tic2_msm /= marginal_tic2_msm.mean() * bin_widths_tic2.sum()
    marginal_tic2_msm = np.append(marginal_tic2_msm, marginal_tic2_msm[-1])
    
    # Calculate negative log of marginal probabilities
    with np.errstate(divide='ignore'):
        neg_log_marginal_tic2 = -np.log(marginal_tic2_msm)
    neg_log_marginal_tic2[~np.isfinite(neg_log_marginal_tic2)] = np.nan

    # Store MSM data
    msm_data.append({
        'cond_pdf': cond_pdf,
        'neg_log_cond_pdf': neg_log_cond_pdf,
        'bin_edges_tic1': bin_edges_tic1,
        'bin_edges_tic2': bin_edges_tic2,
        'marginal_tic1_msm': marginal_tic1_msm,
        'marginal_tic2_msm': marginal_tic2_msm,
        'neg_log_marginal_tic1': neg_log_marginal_tic1,
        'neg_log_marginal_tic2': neg_log_marginal_tic2,
        'phys_lag': int(lag*5)
    })

    # Model
    phys_lag = int(lag*5)
    model_projections = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_init_{str(int(initial_conf)).zfill(6)}_lag_{phys_lag}_nested_{1}_ode_steps_{50}.npy")
    print(f"Lag {phys_lag}: Model projections shape: {model_projections.shape}")
    
    # Store model data
    # Calculate histograms from model projections
    hist_tic1_model, bin_edges_tic1_model = np.histogram(model_projections[:,-1,0], bins=bin_edges_tic1, density=True)
    hist_tic2_model, bin_edges_tic2_model = np.histogram(model_projections[:,-1,1], bins=bin_edges_tic2, density=True)
    
    # Normalize histograms
    hist_tic1_model = hist_tic1_model / (hist_tic1_model.mean() * bin_widths_tic1.sum())
    hist_tic2_model = hist_tic2_model / (hist_tic2_model.mean() * bin_widths_tic2.sum())
    
    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        neg_log_hist_tic1_model = -np.log(hist_tic1_model)
        neg_log_hist_tic2_model = -np.log(hist_tic2_model)
    
    # Replace infinities with NaN
    neg_log_hist_tic1_model[~np.isfinite(neg_log_hist_tic1_model)] = np.nan
    neg_log_hist_tic2_model[~np.isfinite(neg_log_hist_tic2_model)] = np.nan

    # Add hist2d data
    density_2d, xedges_2d, yedges_2d = np.histogram2d(model_projections[:,-1,0], model_projections[:,-1,1], bins=[bin_edges_tic1, bin_edges_tic2], density=True)

    # Store model data
    model_data.append({
        'tic_1_model': model_projections[:,-1,0],
        'tic_2_model': model_projections[:,-1,1],
        'density_2d': density_2d,
        'xedges_2d': xedges_2d,     
        'yedges_2d': yedges_2d, 
        'hist_tic1_model': hist_tic1_model,
        'hist_tic2_model': hist_tic2_model,
        'neg_log_hist_tic1_model': neg_log_hist_tic1_model,
        'neg_log_hist_tic2_model': neg_log_hist_tic2_model,
        'bin_edges_tic1_model': bin_edges_tic1_model,
        'bin_edges_tic2_model': bin_edges_tic2_model,
    })

    # Nested samples
    lag_save = int(phys_lag/5)
    nested_projections = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_init_{str(int(initial_conf)).zfill(6)}_lag_{lag_save}_nested_5_ode_steps_50.npy")

    hist_tic1_model_nested, _ = np.histogram(nested_projections[:,-1,0], bins=bin_edges_tic1, density=True)
    hist_tic2_model_nested, _ = np.histogram(nested_projections[:,-1,1], bins=bin_edges_tic2, density=True)
    hist_tic1_model_nested = hist_tic1_model_nested / (hist_tic1_model_nested.mean() * bin_widths_tic1.sum())
    hist_tic2_model_nested = hist_tic2_model_nested / (hist_tic2_model_nested.mean() * bin_widths_tic2.sum())
    with np.errstate(divide='ignore'):
        neg_log_hist_tic1_model_nested = -np.log(hist_tic1_model_nested)
        neg_log_hist_tic2_model_nested = -np.log(hist_tic2_model_nested)
    neg_log_hist_tic1_model_nested[~np.isfinite(neg_log_hist_tic1_model_nested)] = np.nan
    neg_log_hist_tic2_model_nested[~np.isfinite(neg_log_hist_tic2_model_nested)] = np.nan

    model_data[-1].update({
        'tic_1_model_nested': nested_projections[:,-1,0],
        'tic_2_model_nested': nested_projections[:,-1,1],
        'hist_tic1_model_nested': hist_tic1_model_nested,
        'hist_tic2_model_nested': hist_tic2_model_nested,
        'neg_log_hist_tic1_model_nested': neg_log_hist_tic1_model_nested,
        'neg_log_hist_tic2_model_nested': neg_log_hist_tic2_model_nested,
    })


print(f"Processed {len(lags)} lag times: {[msm_data[i]['phys_lag'] for i in range(len(lags))]}")


In [ ]:
#load harmonic dof data
from tito.utils.mol_geometry import compute_angle
import torch

data_positions = np.load("../other/md.npy")
sampled_positions = np.load("../other/model.npy")

distance_atoms = np.array([[6, 7], [10,11], [0,4]])
distance_titles = ['C-H', 'C-O', 'C-N']
angles_atoms = np.array([[4,6,7], [6, 4, 0], [4,10,11]])
angle_titles = ['C-C-H', 'C-C-N', 'C-C-O']

data_distances = np.linalg.norm(data_positions[:, distance_atoms[:,0]] - data_positions[:, distance_atoms[:,1]], axis=-1)
sampled_distances = np.linalg.norm(sampled_positions[:, distance_atoms[:,0]] - sampled_positions[:, distance_atoms[:,1]], axis=-1)

x1 = data_positions[:, angles_atoms[:,0]]
x2 = data_positions[:, angles_atoms[:,1]]
x3 = data_positions[:, angles_atoms[:,2]]

data_angles = compute_angle(torch.tensor(x1), torch.tensor(x2), torch.tensor(x3)).numpy()
x1 = sampled_positions[:, angles_atoms[:,0]]
x2 = sampled_positions[:, angles_atoms[:,1]]
x3 = sampled_positions[:, angles_atoms[:,2]]
sampled_angles = compute_angle(torch.tensor(x1), torch.tensor(x2), torch.tensor(x3)).numpy()

# Compute histograms and their negative log probabilities
n_bins = 100
distance_hists = {'md': [], 'model': []}
distance_bin_edges = {'md': [], 'model': []}
distance_neg_logps = {'md': [], 'model': []}
distance_bins = []

angle_hists = {'md': [], 'model': []}
angle_bin_edges = {'md': [], 'model': []}
angle_neg_logps = {'md': [], 'model': []}
angle_bins = []

# Process distances
for i in range(data_distances.shape[1]):
    # Find common bin edges for both datasets
    min_val = min(data_distances[:, i].min(), sampled_distances[:, i].min())
    max_val = max(data_distances[:, i].max(), sampled_distances[:, i].max())
    bins = np.linspace(min_val, max_val, n_bins + 1)
    distance_bins.append(bins)
    
    # Compute histograms
    hist_md, bin_edges_ = np.histogram(data_distances[:, i], bins=bins, density=True)
    hist_model, _ = np.histogram(sampled_distances[:, i], bins=bins, density=True)
    
    distance_hists['md'].append(hist_md)
    distance_hists['model'].append(hist_model)
    
    # Compute negative log probabilities with handling of zeros
    with np.errstate(divide='ignore'):
        neg_logp_md = -np.log(hist_md)
        neg_logp_model = -np.log(hist_model)
    
    # Replace infinity with NaN for better plotting
    neg_logp_md[~np.isfinite(neg_logp_md)] = np.nan
    neg_logp_model[~np.isfinite(neg_logp_model)] = np.nan
    
    distance_neg_logps['md'].append(neg_logp_md)
    distance_neg_logps['model'].append(neg_logp_model)
    distance_bin_edges['md'].append(bin_edges_)
    distance_bin_edges['model'].append(bin_edges_)

# Process angles
for i in range(data_angles.shape[1]):
    # Find common bin edges for both datasets
    min_val = min(data_angles[:, i].min(), sampled_angles[:, i].min())
    max_val = max(data_angles[:, i].max(), sampled_angles[:, i].max())
    bins = np.linspace(min_val, max_val, n_bins + 1)
    angle_bins.append(bins)
    
    # Compute histograms
    hist_md, bin_edges_md = np.histogram(data_angles[:, i], bins=bins, density=True)
    hist_model, bin_edges_model = np.histogram(sampled_angles[:, i], bins=bins, density=True)

    angle_hists['md'].append(hist_md)
    angle_hists['model'].append(hist_model)
    angle_bin_edges['md'].append(bin_edges_md)
    angle_bin_edges['model'].append(bin_edges_model)
    
    # Compute negative log probabilities with handling of zeros
    with np.errstate(divide='ignore'):
        neg_logp_md = -np.log(hist_md)
        neg_logp_model = -np.log(hist_model)
    
    # Replace infinity with NaN for better plotting
    neg_logp_md[~np.isfinite(neg_logp_md)] = np.nan
    neg_logp_model[~np.isfinite(neg_logp_model)] = np.nan
    
    angle_neg_logps['md'].append(neg_logp_md)
    angle_neg_logps['model'].append(neg_logp_model)


In [ ]:
#load energy data

md = np.load("../other/energies_md_ts.npy")[:, [1,2,3,-1]]
md = np.sum(md, axis=1)
model = np.load("../other/energies_model_ts.npy")[:, [1,2,3,-1]]
model = np.sum(model, axis=1)
upper_bound = np.percentile(model, 99)
model_filtered = model[model <= upper_bound]


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 14))

# Overall 1 row × 2 column structure
outer = gridspec.GridSpec(1, 2, width_ratios=[2., 1], wspace=0.25)

# === Left side (stack of 5 rows, each with 4 columns) ===
left = gridspec.GridSpecFromSubplotSpec(
    5, 4, subplot_spec=outer[0], wspace=0.5, hspace=0.2, width_ratios=[1, 0.05, 1, 1] #extra column for colorbar
)

# Example: access axes like this
axes_left = []
for i in range(5):  # 5 rows
    row_axes = []
    for j in range(4):  # 3 columns
        ax = fig.add_subplot(left[i, j])
        row_axes.append(ax)
    axes_left.append(row_axes)

# === Right side (2 rows: top with 2×3 grid, bottom with single panel) ===
right = gridspec.GridSpecFromSubplotSpec(
    2, 1, subplot_spec=outer[1], height_ratios=[3, 1], hspace=0.2
)

# Top grid: 2 rows × 3 columns
top = gridspec.GridSpecFromSubplotSpec(
    2, 3, subplot_spec=right[0], wspace=0.35, hspace=0.3
)
axes_top = [fig.add_subplot(top[i, j]) for i in range(2) for j in range(3)]

# Bottom single panel
ax_bottom = fig.add_subplot(right[1])

# Mark this as subfigure (a)
fig.text(0.0575, 0.9, 'A', fontproperties=font_prop, fontweight='bold')

# Mark top right as subfigure (b)
fig.text(0.6, 0.9, 'B', fontproperties=font_prop, fontweight='bold')

# Mark bottom right as subfigure (c)
fig.text(0.6, 0.3, 'C', fontproperties=font_prop, fontweight='bold')

#left panel
# Create a dictionary to map lag times to their string representation
lag_labels = {10: '$50$ $ps$', 20: '$100$ $ps$', 100: '$500$ $ps$', 200: '$1000$ $ps$', 1000: '$5000$ $ps$'}

# Set up nested samples structure for left panel - each panel gets a label
for i_lag, lag in enumerate(lags):
    # Add lag time label to the left of each row
    if i_lag < len(lags):
        # Adjust position to align with the first plot in each row
        row_label_x = axes_left[i_lag][0].get_position().x0 - 0.05
        row_label_y = (axes_left[i_lag][0].get_position().y0 + 
                       axes_left[i_lag][0].get_position().y1) / 2
        
        # Add the label
        fig.text(row_label_x, row_label_y, f"{lag_labels[lag]}", rotation=90, 
                 va='center', ha='right', fontproperties=font_prop)


# Plot TICA marginals in the left panel
for i_lag in range(len(lags)):
    lag = lags[i_lag]
    
    # Get MSM and model data for this lag
    msm_data_lag = msm_data[i_lag]
    model_data_lag = model_data[i_lag]
    
    # Plot MSM conditional PDFs
    im = axes_left[i_lag][0].pcolormesh(msm_data_lag['bin_edges_tic1'], 
                             msm_data_lag['bin_edges_tic2'],
                             msm_data_lag['neg_log_cond_pdf'], 
                             cmap='viridis_r')
    
    cbar = fig.colorbar(im, cax=axes_left[i_lag][1])
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=10)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())
    pos = axes_left[i_lag][1].get_position()
    axes_left[i_lag][1].set_position([pos.x0 - 0.035, pos.y0, pos.width, pos.height])

    #cbar = fig.colorbar(im, ax=axes_left[i_lag][1])
    #cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop)
    #cbar.ax.tick_params(labelsize=font["size"])
    
    # Add contour lines to highlight the distribution structure
    cs = axes_left[i_lag][0].contour(np.log(model_data_lag['density_2d']).transpose(), extent=[model_data_lag['xedges_2d'][0], 
                                               model_data_lag['xedges_2d'][-1], 
                                               model_data_lag['yedges_2d'][0], 
                                               model_data_lag['yedges_2d'][-1]], 
                          colors=palette_colors[1], alpha=1., levels=[-7, -5, -3, -2])

    # Add a small cross to indicate the initial configuration
    axes_left[i_lag][0].plot(initial_ticas[0], initial_ticas[1], 'x', 
                              color='red', markersize=8, markeredgewidth=2, zorder=1)
    
    axes_left[i_lag][0].tick_params(axis='both', which='major', labelsize=font["size"])
    
    # # Plot model histogram as scatter
    # axes_left[i_lag][0].scatter(model_data_lag['tic_1_model'], model_data_lag['tic_2_model'], 
    #                   s=1, alpha=0.5, c=palette_colors[1])
    
    # axes_left[i_lag][0].set_title(f'Lag = {msm_data_lag["phys_lag"]} ps', fontproperties=font_prop)
    
    # Plot marginal TIC1
    axes_left[i_lag][2].step(msm_data_lag['bin_edges_tic1'], msm_data_lag['neg_log_marginal_tic1'], where='post',
                   color=palette_colors[0], label='MD')
    axes_left[i_lag][2].step(model_data_lag['bin_edges_tic1_model'][:-1], model_data_lag['neg_log_hist_tic1_model'], 
                             where='post', color=palette_colors[1], label='TITO')
    axes_left[i_lag][2].step(model_data_lag['bin_edges_tic1_model'][:-1], model_data_lag['neg_log_hist_tic1_model_nested'], 
                             where='post', color='orange', label='TITO nested')
    axes_left[i_lag][2].tick_params(axis='both', which='major', labelsize=font["size"])

    #axes_left[i_lag][2].set_yscale('log')
    
    # Plot marginal TIC2
    axes_left[i_lag][3].step(msm_data_lag['bin_edges_tic2'], msm_data_lag['neg_log_marginal_tic2'], where='post',
                   color=palette_colors[0], label='MD')
    axes_left[i_lag][3].step(model_data_lag['bin_edges_tic2_model'][:-1], model_data_lag['neg_log_hist_tic2_model'], 
                             where='post', color=palette_colors[1], label='TITO')
    axes_left[i_lag][3].step(model_data_lag['bin_edges_tic2_model'][:-1], model_data_lag['neg_log_hist_tic2_model_nested'], 
                             where='post', color='orange', label='TITO nested')
    axes_left[i_lag][3].tick_params(axis='both', which='major', labelsize=font["size"])
    

    #axes_left[i_lag][3].set_yscale('log')

# Add labels for the first row only to avoid redundancy
axes_left[-1][0].set_xlabel('TIC1', fontproperties=font_prop)
axes_left[2][0].set_ylabel('TIC2', fontproperties=font_prop)
axes_left[-1][2].set_xlabel('TIC1', fontproperties=font_prop)
axes_left[-1][3].set_xlabel('TIC2', fontproperties=font_prop)

# Add a single legend for all TICA plots
handles, labels = axes_left[0][2].get_legend_handles_labels()

fig.legend(handles, labels, loc="lower left", bbox_to_anchor=(0.3, 0.04), 
           prop=font_prop, ncol=3)



#top panel
# Top panel - Plot distances in first row
for i in range(3):
    axes_top[i].step(distance_bin_edges['md'][i][:-1]*10, distance_neg_logps['md'][i], where='post', label='MD', color=palette_colors[0])
    axes_top[i].step(distance_bin_edges['model'][i][:-1]*10, distance_neg_logps['model'][i], where='post', label='TITO', color=palette_colors[1])
    axes_top[i].set_title(distance_titles[i], fontproperties=font_prop)
    axes_top[i].tick_params(axis='both', which='major', labelsize=font["size"])
    

# Plot angles in second row
for i in range(3):
    axes_top[i+3].step(angle_bin_edges['md'][i][:-1], angle_neg_logps['md'][i], where='post', label='MD', color=palette_colors[0])
    axes_top[i+3].step(angle_bin_edges['model'][i][:-1], angle_neg_logps['model'][i], where='post', label='TITO', color=palette_colors[1])
    axes_top[i+3].set_title(angle_titles[i], fontproperties=font_prop)
    axes_top[i+3].tick_params(axis='both', which='major', labelsize=font["size"])

# Add shared labels
axes_top[1].set_xlabel('Distance [Å]', fontproperties=font_prop)
axes_top[4].set_xlabel('Angle [rad]', fontproperties=font_prop)
axes_top[0].set_ylabel('Free energy $[k_BT]$', fontproperties=font_prop)
axes_top[3].set_ylabel('Free energy $[k_BT]$', fontproperties=font_prop)


#bottom panel

# Plot histograms
md_hist = ax_bottom.hist(md, bins=50, alpha=0.7, label='MD', color=palette_colors[0],  histtype='step', density=True);
ax_bottom.hist(model_filtered, bins=50, alpha=0.7, label='TITO', color=palette_colors[1], histtype='step', density=True);

# Add labels and legend
ax_bottom.set_xlabel('Non-harmonic energy $[k_BT]$', fontproperties=font_prop)
ax_bottom.set_ylabel('Density', fontproperties=font_prop)
ax_bottom.tick_params(axis='both', which='major', labelsize=font["size"])

plt.show()


In [ ]:
path = f"../plots/paper_figures/main/fig4_marginals.pdf"
if not osp.exists(osp.dirname(path)):
    os.makedirs(osp.dirname(path))
fig.savefig(path, transparent=True)


## Figure 5: Extrapolation


In [ ]:
import os.path as osp
import os
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.font_manager as fm
from matplotlib import gridspec

font_size = 14
font = {#'family': 'Liberation Sans',
    'weight' : 'normal',
    'size'   : font_size}


#matplotlib.rc('font', **font)

font_path = None
font_prop = fm.FontProperties(fname=font_path, size=font["size"], weight='normal')

palette_colors = ["#830051", "#68D2DF", "#F0AB00", "#83928E", "#cad502", "#e6007b","#515151"] #purple, blue, orange, grey, green, pink

In [ ]:
# Labels for each peptide row
row_labels = [
    ("Penta-peptide", "STVSK"),
    ("Hexa-peptide", "IASSNI"),
    ("Hepta-peptide", "MAECPFW"),
    ("Octa-peptide", "FTQPEVSL")
]

md = []
tito = []
edges = []
md_timescales = []
tito_timescales = []
vamp2_gap = []

model = "lxf356pw" #"jcxoitlc" # uniform: 57hk8qzx, lxf356pw. exp: ktgl41l5, dg5fl7wv
lag = 5000 #512  
nested_samples = 5 #2
ode_steps = 100
batchsize = 512 #100
#n_replicas = 150
#n_steps = nested_samples + 1
step_to_use = 1
tica_dim = 2

for family, system in row_labels:
    md_projections = np.load(f"{RESULTS_DIR}/custom/{system}/md/tica_projections.npy")
    model_projections = np.load(f"{RESULTS_DIR}/custom/{system}/{model}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")#_ode_steps_{ode_steps} short_params/

    #model_projections = model_projections.reshape(batchsize, n_replicas, n_steps, tica_dim)
    #model_projections = model_projections[:,:,step_to_use,:].reshape(-1, model_projections.shape[-1])
    model_projections = model_projections[:,step_to_use,:].reshape(-1, model_projections.shape[-1])
    md_projections = md_projections.reshape(-1, md_projections.shape[-1])
    #model_projections = model_projections.reshape(-1, model_projections.shape[-1])

    # Subsample md projections to match model projections size
    np.random.seed(42)  # Set seed for reproducibility
    min_samples = min(len(md_projections), len(model_projections))
    indices_md = np.random.choice(len(md_projections), size=min_samples, replace=False)
    md_projections_subsampled = md_projections[indices_md]
    indices_model = np.random.choice(len(model_projections), size=min_samples, replace=False)
    model_projections_subsampled = model_projections[indices_model]

    # Calculate histograms from re projections
    counts_md, xedges, yedges = np.histogram2d(md_projections_subsampled[:,0], md_projections_subsampled[:,1], bins=100)
    counts_md = counts_md.T  #  Transpose to match the orientation of pcolormesh    
    counts_model, _, _ = np.histogram2d(model_projections[:,0], model_projections[:,1], bins=[xedges, yedges])
    counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh
    edges.append((xedges, yedges))
    # Normalize histograms
    pmf_md = counts_md / np.sum(counts_md)
    pmf_model = counts_model / np.sum(counts_model)

    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        fe_md = -np.log(pmf_md)
        fe_model = -np.log(pmf_model)   
    # Replace infinities with NaN
    fe_md[~np.isfinite(fe_md)] = np.nan
    fe_model[~np.isfinite(fe_model)] = np.nan
    md.append(fe_md)
    tito.append(fe_model)


    md_svs = np.load(f"{RESULTS_DIR}/custom/{system}/md/vamp_singular_values.npy")
    model_svs = np.load(f"{RESULTS_DIR}/custom/{system}/{model}/vamp_singular_values_random_init_lag_{int(lag)}_nested_{nested_samples}_ode_steps_{ode_steps}.npy") #short_params. _ode_steps_{ode_steps}

    #vamp_gap = np.load()
    vamp2_gap.append(np.load(f"{RESULTS_DIR}/custom/{system}/{model}/vamp_gap_random_init_lag_{int(lag)}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")) ##short_params. _ode_steps_{ode_steps}
    #compute implied timescales
    md_ts= -lag/np.log(md_svs)
    ts_indices = np.where(md_ts > lag/5)
    md_ts = md_ts[ts_indices]
    md_timescales.append(md_ts)
    model_ts=-lag/np.log(model_svs)
    tito_timescales.append(model_ts[ts_indices])

    

In [ ]:
model_projections_subsampled.shape, md_projections_subsampled.shape

In [ ]:
n_blocks = 4

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(13, 16))
outer = gridspec.GridSpec(n_blocks, 1, hspace=0.001) #, wspace=0.35

# Loop over blocks
for block_idx in range(n_blocks):
    # --- Left column (MD density)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 5,  # extra column for colorbar
        subplot_spec=outer[block_idx],
        hspace=0.0,
        wspace=0.3,
        width_ratios=[0.001, 1, 1, 0.25, 1]  # spacer before colorbar, then ax3
    )
    ax1 = fig.add_subplot(inner[0,1])
    #ax1 = fig.add_subplot(outer[block_idx, 0])
    
    if block_idx == 0:
        ax1.set_title("MD", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax1.set_xlabel('TIC 1', fontproperties=font_prop)
    ax1.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)
    # Apply font properties to tick labels
    ax1.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    # Set 5 ticks for both axes
    ax1.locator_params(axis='both', nbins=5)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    

    ax2 = fig.add_subplot(inner[0, 2])
    if block_idx == 0:
        ax2.set_title("TITO", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax2.set_xlabel('TIC 1', fontproperties=font_prop)
    #ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)
    # Apply font properties to tick labels
    ax2.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax2.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax2.get_position().x1 + 0.01, 
                            ax2.get_position().y0, 
                            0.01, 
                            ax2.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())

    # --- Right column (Timescale scatter)
    ax3 = fig.add_subplot(inner[0, 4])
    ax3.set_box_aspect(1)

    # Plot timescale scatter
    ax3.scatter(md_timescales[block_idx], tito_timescales[block_idx], color=palette_colors[1], s=100, zorder=2)
    rel_error = np.mean(np.abs((tito_timescales[block_idx][:10] - md_timescales[block_idx][:10]) / md_timescales[block_idx][:10]))

    #Set log scale
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_box_aspect(1)
    # Apply font properties to tick labels
    ax3.tick_params(axis='both', which='major', labelsize=font_prop.get_size())

    #Set equal range in both axis
    # Get current limits
    xmin, xmax = ax3.get_xlim()
    ymin, ymax = ax3.get_ylim()

    # Find the combined min and max
    low = min(xmin, ymin)
    high = max(xmax, ymax)

    ax3.plot(np.arange(low, high, (high-low)/10), np.arange(low, high, (high-low)/10), color=palette_colors[0], linestyle='dashed',
        linewidth=2, markersize=12, zorder=1, label='y=x')

    # Apply to both ax3
    ax3.set_xlim(low, high)
    ax3.set_ylim(low, high)

    ax3.plot([lag for _ in range(10)], np.arange(low, lag, (lag-low)/10), color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)
    ax3.plot(np.arange(low, lag, (lag-low)/10), [lag for _ in range(10)], color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)

    ax3.text(0.02, 0.9, f'Relative error = {rel_error:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    ax3.text(0.02, 0.8, f'VAMP-2 gap  = {vamp2_gap[block_idx]:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    plt.tick_params(axis='both', which='major', labelsize=font["size"])

    if block_idx == n_blocks - 1:
        ax3.set_xlabel('Time-scale MD [$ps$]', fontproperties=font_prop, labelpad=1)
    ax3.set_ylabel('Time-scale model [$ps$]', fontproperties=font_prop, labelpad=1)

    # --- Add row annotation on the left
    row = outer[block_idx, :]
    y_mid = (row.get_position(fig).y0 + row.get_position(fig).y1) / 2
    fig.text(0.07, y_mid, row_labels[block_idx][0],
             va="center", ha="left",  rotation=90, fontproperties=font_prop)
    fig.text(0.07+0.02, y_mid, row_labels[block_idx][1],
             va="center", ha="left", rotation=90, fontproperties=font_prop)

plt.tight_layout(rect=[0.05, 0, 1, 1])  # leave more space on the left for labels
plt.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/main/fig5_extrapolation.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/main/fig5_extrapolation.pdf", transparent=True)

In [ ]:
import os.path as osp
import os
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.font_manager as fm
from matplotlib import gridspec

font_size = 18
font = {#'family': 'Liberation Sans',
    'weight' : 'normal',
    'size'   : font_size}


#matplotlib.rc('font', **font)

font_path = None
font_prop = fm.FontProperties(fname=font_path, size=font["size"], weight='normal')

palette_colors = ["#830051", "#68D2DF", "#F0AB00", "#83928E", "#cad502", "#e6007b","#515151"] #purple, blue, orange, grey, green, pink

In [ ]:
n_blocks = 4

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(13, 16))
outer = gridspec.GridSpec(n_blocks, 1, hspace=0.001) #, wspace=0.35

# Loop over blocks
for block_idx in range(n_blocks):
    # --- Left column (MD density)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 5,  # extra column for colorbar
        subplot_spec=outer[block_idx],
        hspace=0.0,
        wspace=0.4,
        width_ratios=[0.001, 1, 1, 0.25, 1]  # spacer before colorbar, then ax3
    )
    ax1 = fig.add_subplot(inner[0,1])
    #ax1 = fig.add_subplot(outer[block_idx, 0])
    
    if block_idx == 0:
        ax1.set_title("MD", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax1.set_xlabel('Collective variable 1', fontproperties=font_prop)
    ax1.set_ylabel('Collective variable 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)
    # Apply font properties to tick labels
    ax1.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    # Set 5 ticks for both axes
    ax1.locator_params(axis='both', nbins=5)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    

    ax2 = fig.add_subplot(inner[0, 2])
    if block_idx == 0:
        ax2.set_title("TITO", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax2.set_xlabel('Collective variable 1', fontproperties=font_prop)
    #ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)
    # Apply font properties to tick labels
    ax2.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax2.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax2.get_position().x1 + 0.01, 
                            ax2.get_position().y0, 
                            0.01, 
                            ax2.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())

    # --- Right column (Timescale scatter)
    ax3 = fig.add_subplot(inner[0, 4])
    ax3.set_box_aspect(1)

    # Plot timescale scatter
    ax3.scatter(md_timescales[block_idx], tito_timescales[block_idx], color=palette_colors[1], s=100, zorder=2)
    rel_error = np.mean(np.abs((tito_timescales[block_idx][:10] - md_timescales[block_idx][:10]) / md_timescales[block_idx][:10]))

    #Set log scale
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_box_aspect(1)
    # Apply font properties to tick labels
    ax3.tick_params(axis='both', which='major', labelsize=font_prop.get_size())

    #Set equal range in both axis
    # Get current limits
    xmin, xmax = ax3.get_xlim()
    ymin, ymax = ax3.get_ylim()

    # Find the combined min and max
    low = min(xmin, ymin)
    high = max(xmax, ymax)

    ax3.plot(np.arange(low, high, (high-low)/10), np.arange(low, high, (high-low)/10), color=palette_colors[0], linestyle='dashed',
        linewidth=2, markersize=12, zorder=1, label='y=x')

    # Apply to both ax3
    ax3.set_xlim(low, high)
    ax3.set_ylim(low, high)

    ax3.plot([lag for _ in range(10)], np.arange(low, lag, (lag-low)/10), color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)
    ax3.plot(np.arange(low, lag, (lag-low)/10), [lag for _ in range(10)], color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)

    # ax3.text(0.02, 0.9, f'Relative error = {rel_error:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    # ax3.text(0.02, 0.8, f'VAMP-2 gap  = {vamp2_gap[block_idx]:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    # plt.tick_params(axis='both', which='major', labelsize=font["size"])

    if block_idx == n_blocks - 1:
        ax3.set_xlabel('Time-scale MD [$ps$]', fontproperties=font_prop, labelpad=1)
    ax3.set_ylabel('Time-scale model [$ps$]', fontproperties=font_prop, labelpad=1)

    # --- Add row annotation on the left
    row = outer[block_idx, :]
    y_mid = (row.get_position(fig).y0 + row.get_position(fig).y1) / 2
    fig.text(0.05, y_mid, row_labels[block_idx][0],
             va="center", ha="left",  rotation=90, fontproperties=font_prop)
    fig.text(0.05+0.02, y_mid, row_labels[block_idx][1],
             va="center", ha="left", rotation=90, fontproperties=font_prop)

plt.tight_layout(rect=[0.05, 0, 1, 1])  # leave more space on the left for labels
plt.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/main/fig5_extrapolation_simon.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/main/fig5_extrapolation_simon.pdf", transparent=True)

## Appendix

In [ ]:
import os.path as osp
import os
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.font_manager as fm
from matplotlib import gridspec

font_size = 14
font = {#'family': 'Liberation Sans',
    'weight' : 'normal',
    'size'   : font_size}


#matplotlib.rc('font', **font)

font_path = None
font_prop = fm.FontProperties(fname=font_path, size=font["size"], weight='normal')

palette_colors = ["#830051", "#68D2DF", "#F0AB00", "#83928E", "#cad502", "#e6007b","#515151"] #purple, blue, orange, grey, green, pink

### Examples different scales of longest process

In [ ]:
mol_indices = [7,0,3]

md = []
tito = []
edges = []
md_timescales = []
tito_timescales = []
vamp2_gap = []


data_set = "mdqm9"
model = "nfksixdq" # uniform: gn10le53, nfksixdq . exp: g2myka6s, wuck6wrs
split = "test"
lag = 57
nested_samples = 640

tica_dim = 2

for mol_index in mol_indices:
    md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
    model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}.npy")

    md_projections = md_projections.reshape(-1, md_projections.shape[-1])
    model_projections = model_projections.reshape(-1, model_projections.shape[-1])

    # Subsample model projections to match md projections size
    np.random.seed(42)  # Set seed for reproducibility
    indices = np.random.choice(len(model_projections), size=len(md_projections), replace=False)
    model_projections_subsampled = model_projections[indices]

    # Calculate histograms from re projections
    counts_md, xedges, yedges = np.histogram2d(md_projections[:,0], md_projections[:,1], bins=100)
    counts_md = counts_md.T  #  Transpose to match the orientation of pcolormesh
    counts_model, _, _ = np.histogram2d(model_projections_subsampled[:,0], model_projections_subsampled[:,1], bins=[xedges, yedges])
    counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh
    edges.append((xedges, yedges))
    # Normalize histograms
    pmf_md = counts_md / np.sum(counts_md)
    pmf_model = counts_model / np.sum(counts_model)

    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        fe_md = -np.log(pmf_md)
        fe_model = -np.log(pmf_model)   
    # Replace infinities with NaN
    fe_md[~np.isfinite(fe_md)] = np.nan
    fe_model[~np.isfinite(fe_model)] = np.nan
    md.append(fe_md)
    tito.append(fe_model)


    md_svs = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values.npy")
    #vamp_gap = np.load()
    model_svs = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values_random_init_lag_{int(lag)}_nested_{nested_samples}.npy")
    vamp2_gap.append(np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_gap_random_init_lag_{int(lag)}_nested_{nested_samples}.npy"))
    #compute implied timescales
    md_ts= -lag/np.log(md_svs)
    ts_indices = np.where(md_ts > lag/5)
    md_ts = md_ts[ts_indices]
    md_timescales.append(md_ts)
    model_ts=-lag/np.log(model_svs)
    tito_timescales.append(model_ts[ts_indices])

    

In [ ]:
n_blocks = 3

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(14, 12))
outer = gridspec.GridSpec(n_blocks, 1, hspace=0.001) #, wspace=0.35

# Loop over blocks
for block_idx in range(n_blocks):
    # --- Left column (MD density)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4,  # extra column for colorbar
        subplot_spec=outer[block_idx],
        hspace=0.1,
        wspace=0.3,
        width_ratios=[ 1, 1, 0.25, 1]  # spacer before colorbar, then ax3
    )
    ax1 = fig.add_subplot(inner[0,0])
    #ax1 = fig.add_subplot(outer[block_idx, 0])
    
    if block_idx == 0:
        ax1.set_title("MD", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax1.set_xlabel('TIC 1', fontproperties=font_prop)
    ax1.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)
    # Apply font properties to tick labels
    ax1.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    # Set 5 ticks for both axes
    ax1.locator_params(axis='both', nbins=5)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    

    ax2 = fig.add_subplot(inner[0, 1])
    if block_idx == 0:
        ax2.set_title("TITO", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax2.set_xlabel('TIC 1', fontproperties=font_prop)
    #ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)
    # Apply font properties to tick labels
    ax2.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax2.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax2.get_position().x1 + 0.01, 
                            ax2.get_position().y0, 
                            0.01, 
                            ax2.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())

    # --- Right column (Timescale scatter)
    ax3 = fig.add_subplot(inner[0, 3])

    # Plot timescale scatter
    ax3.scatter(md_timescales[block_idx], tito_timescales[block_idx], color=palette_colors[1], s=100, zorder=2)
    rel_error = np.mean(np.abs((tito_timescales[block_idx][:10] - md_timescales[block_idx][:10]) / md_timescales[block_idx][:10]))

    #Set log scale
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_box_aspect(1)
    # Apply font properties to tick labels
    if block_idx == 2:
        ax3.tick_params(axis='both', which='major', labelsize=font_prop.get_size())

    #Set equal range in both axis
    # Get current limits
    xmin, xmax = ax3.get_xlim()
    ymin, ymax = ax3.get_ylim()

    # Find the combined min and max
    low = min(xmin, ymin)
    high = max(xmax, ymax)

    ax3.plot(np.arange(low, high, (high-low)/10), np.arange(low, high, (high-low)/10), color=palette_colors[0], linestyle='dashed',
        linewidth=2, markersize=12, zorder=1, label='y=x')

    # Apply to both ax3
    ax3.set_xlim(low, high)
    ax3.set_ylim(low, high)

    ax3.plot([lag for _ in range(10)], np.arange(low, lag, (lag-low)/10), color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)
    ax3.plot(np.arange(low, lag, (lag-low)/10), [lag for _ in range(10)], color=palette_colors[-1], linestyle='dotted',
        linewidth=2, markersize=12, zorder=1)

    ax3.text(0.02, 0.9, f'Relative error = {rel_error:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    ax3.text(0.02, 0.8, f'VAMP-2 gap  = {vamp2_gap[block_idx]:.3f}', transform=ax3.transAxes, fontproperties=font_prop)
    plt.tick_params(axis='both', which='major', labelsize=font["size"])

    if block_idx == n_blocks - 1:
        ax3.set_xlabel('Time-scale MD [$ps$]', fontproperties=font_prop, labelpad=1)
    ax3.set_ylabel('Time-scale model [$ps$]', fontproperties=font_prop, labelpad=1)


plt.tight_layout(rect=[0.05, 0, 1, 1])  # leave more space on the left for labels
plt.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/appendix/multi_scale.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/appendix/multi_scale.pdf", transparent=True)

### Examples for different JSDs


In [ ]:
mol_indices = [27,26,88]

md = []
re = []
tito = []
edges = []
md_timescales = []
tito_timescales = []
vamp2_gap = []


data_set = "mdqm9"
model = "nfksixdq" # uniform: gn10le53, nfksixdq . exp: g2myka6s, wuck6wrs
split = "test"
lag = 57
nested_samples = 640

tica_dim = 2

for mol_index in mol_indices:
    md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
    re_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/re/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_re.npy")   
    model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}.npy")


    md_projections = md_projections.reshape(-1, md_projections.shape[-1])
    re_projections = re_projections.reshape(-1, re_projections.shape[-1])
    model_projections = model_projections.reshape(-1, model_projections.shape[-1])

    # Subsample model projections to match md projections size
    np.random.seed(42)  # Set seed for reproducibility
    min_samples = min(len(md_projections), len(model_projections), len(re_projections))
    indices_model = np.random.choice(len(model_projections), size=len(md_projections), replace=False)
    model_projections_subsampled = model_projections[indices_model]
    indices_md = np.random.choice(len(md_projections), size=min_samples, replace=False)
    md_projections_subsampled = md_projections[indices_md]
    indices_re = np.random.choice(len(re_projections), size=min_samples, replace=False)
    re_projections = re_projections[indices_re]

    # Calculate histograms from re projections
    counts_md, xedges, yedges = np.histogram2d(md_projections[:,0], md_projections[:,1], bins=100)
    counts_md = counts_md.T  #  Transpose to match the orientation of pcolormesh
    counts_re, xedges, yedges = np.histogram2d(re_projections[:,0], re_projections[:,1], bins=100)
    counts_re = counts_re.T 
    counts_model, _, _ = np.histogram2d(model_projections_subsampled[:,0], model_projections_subsampled[:,1], bins=[xedges, yedges])
    counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh
    edges.append((xedges, yedges))
    # Normalize histograms
    pmf_md = counts_md / np.sum(counts_md)
    pmf_re = counts_re / np.sum(counts_re)
    pmf_model = counts_model / np.sum(counts_model)

    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        fe_md = -np.log(pmf_md)
        fe_re = -np.log(pmf_re)
        fe_model = -np.log(pmf_model)   
    # Replace infinities with NaN
    fe_md[~np.isfinite(fe_md)] = np.nan
    fe_model[~np.isfinite(fe_model)] = np.nan
    md.append(fe_md)
    re.append(fe_re)
    tito.append(fe_model)


In [ ]:
n_blocks = 3

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(14, 12))
outer = gridspec.GridSpec(n_blocks, 1, hspace=0.001) #, wspace=0.35

# Loop over blocks
for block_idx in range(n_blocks):
    # --- Left column (MD density)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4,  # extra column for colorbar
        subplot_spec=outer[block_idx],
        hspace=0.1,
        wspace=0.3,
        width_ratios=[ 1, 1, 1, 0.25]  # spacer before colorbar, then ax3
    )
    ax1 = fig.add_subplot(inner[0,0])
    #ax1 = fig.add_subplot(outer[block_idx, 0])
    
    if block_idx == 0:
        ax1.set_title("MD", fontproperties=font_prop)

    ax1.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)
    # Apply font properties to tick labels
    ax1.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    # Set 5 ticks for both axes
    ax1.locator_params(axis='both', nbins=5)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    

    ax2 = fig.add_subplot(inner[0, 1])
    if block_idx == 0:
        ax2.set_title("RE", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax2.set_xlabel('TIC 1', fontproperties=font_prop)
    #ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)
    # Apply font properties to tick labels
    ax2.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax2.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], re[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD

    # --- Right column (Timescale scatter)
    ax3 = fig.add_subplot(inner[0, 2])
    if block_idx == 0:
        ax3.set_title("TITO", fontproperties=font_prop)
    ax3.set_box_aspect(1)
    # Apply font properties to tick labels
    ax3.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax3.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax3.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax3.get_position().x1 + 0.01, 
                            ax3.get_position().y0, 
                            0.01, 
                            ax3.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())

plt.tight_layout(rect=[0.05, 0, 1, 1])  # leave more space on the left for labels
plt.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/appendix/examples_jsd.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/appendix/examples_jsd.pdf", transparent=True)

### TITO better than MD wrt RE


In [ ]:
mol_indices = [86, 81, 61] #13, 29

md = []
re = []
tito = []
edges = []
md_timescales = []
tito_timescales = []
vamp2_gap = []


data_set = "mdqm9"
model = "nfksixdq" # uniform: gn10le53, nfksixdq . exp: g2myka6s, wuck6wrs
split = "test"
lag = 57
nested_samples = 640

tica_dim = 2

for mol_index in mol_indices:
    md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
    re_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/re/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_re.npy")   
    model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}.npy")


    md_projections = md_projections.reshape(-1, md_projections.shape[-1])
    re_projections = re_projections.reshape(-1, re_projections.shape[-1])
    model_projections = model_projections.reshape(-1, model_projections.shape[-1])

    # Subsample model projections to match md projections size
    np.random.seed(42)  # Set seed for reproducibility
    min_sample_size = min(len(md_projections), len(re_projections), len(model_projections))
    indices_model = np.random.choice(len(model_projections), size=min_sample_size, replace=False)
    model_projections_subsampled = model_projections[indices_model]
    indices_md = np.random.choice(len(md_projections), size=min_sample_size, replace=False)
    md_projections = md_projections[indices_md]
    indices_re = np.random.choice(len(re_projections), size=min_sample_size, replace=False)
    re_projections = re_projections[indices_re]

    # Calculate histograms from re projections
    counts_md, xedges, yedges = np.histogram2d(md_projections[:,0], md_projections[:,1], bins=100)
    counts_md = counts_md.T  #  Transpose to match the orientation of pcolormesh
    counts_re, xedges, yedges = np.histogram2d(re_projections[:,0], re_projections[:,1], bins=100)
    counts_re = counts_re.T 
    counts_model, _, _ = np.histogram2d(model_projections_subsampled[:,0], model_projections_subsampled[:,1], bins=[xedges, yedges])
    counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh
    edges.append((xedges, yedges))
    # Normalize histograms
    pmf_md = counts_md / np.sum(counts_md)
    pmf_re = counts_re / np.sum(counts_re)
    pmf_model = counts_model / np.sum(counts_model)

    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        fe_md = -np.log(pmf_md)
        fe_re = -np.log(pmf_re)
        fe_model = -np.log(pmf_model)   
    # Replace infinities with NaN
    fe_md[~np.isfinite(fe_md)] = np.nan
    fe_model[~np.isfinite(fe_model)] = np.nan
    md.append(fe_md)
    re.append(fe_re)
    tito.append(fe_model)


In [ ]:
n_blocks = 3

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(14, 12))
outer = gridspec.GridSpec(n_blocks, 1, hspace=0.001) #, wspace=0.35

# Loop over blocks
for block_idx in range(n_blocks):
    # --- Left column (MD density)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4,  # extra column for colorbar
        subplot_spec=outer[block_idx],
        hspace=0.1,
        wspace=0.3,
        width_ratios=[ 1, 1, 1, 0.25]  # spacer before colorbar, then ax3
    )

    ax2 = fig.add_subplot(inner[0, 1])
    if block_idx == 0:
        ax2.set_title("RE", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax2.set_xlabel('TIC 1', fontproperties=font_prop)
    #ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)
    # Apply font properties to tick labels
    ax2.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax2.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], re[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    

    xlim, ylim = ax2.get_xlim(), ax2.get_ylim()
    ax1 = fig.add_subplot(inner[0,0])
    #ax1 = fig.add_subplot(outer[block_idx, 0])
    
    if block_idx == 0:
        ax1.set_title("MD", fontproperties=font_prop)

    ax1.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)
    # Apply font properties to tick labels
    ax1.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    # Set 5 ticks for both axes
    ax1.locator_params(axis='both', nbins=5)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    ax1.set_xlim(xlim)
    ax1.set_ylim(ylim)
    
    # --- Right column (Timescale scatter)
    ax3 = fig.add_subplot(inner[0, 2])
    if block_idx == 0:
        ax3.set_title("TITO", fontproperties=font_prop)
    ax3.set_box_aspect(1)
    # Apply font properties to tick labels
    ax3.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax3.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax3.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax3.get_position().x1 + 0.01, 
                            ax3.get_position().y0, 
                            0.01, 
                            ax3.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())

    ax3.set_xlim(xlim)
    ax3.set_ylim(ylim)

plt.tight_layout(rect=[0.05, 0, 1, 1])  # leave more space on the left for labels
plt.show()


In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/appendix/examples_tito_better.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/appendix/examples_tito_better.pdf", transparent=True)

### TITO "over-predicts" ns stable states


In [ ]:
mol_indices = [30, 32, 25, 36]

md = []
re = []
tito = []
mdft = []
edges = []
md_timescales = []
tito_timescales = []
vamp2_gap = []


data_set = "mdqm9"
model = "nfksixdq" # uniform: gn10le53, nfksixdq . exp: g2myka6s, wuck6wrs
split = "test"
lag = 1000
nested_samples = 1000
ode_steps = 40
ft_ps = 1000

tica_dim = 2

for mol_index in mol_indices:
    md_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/md/{split}/mol_{str(mol_index).zfill(5)}/tica_projections.npy")
    re_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/re/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_re.npy")   
    model_projections = np.load(f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_random_init_lag_{lag}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")
    mdft_projections_path = f"{RESULTS_DIR}/{data_set}/version_0/{model}/{split}/mol_{str(mol_index).zfill(5)}/tica_projections_ft_positions_{ft_ps}_ps_random_init_lag_{lag}_nested_{nested_samples}_ode_steps_{ode_steps}.npy"
    mdft_projections = np.load(mdft_projections_path)


    md_projections = md_projections.reshape(-1, md_projections.shape[-1])
    re_projections = re_projections.reshape(-1, re_projections.shape[-1])
    model_projections = model_projections.reshape(-1, model_projections.shape[-1])
    mdft_projections = mdft_projections.reshape(-1, mdft_projections.shape[-1])

    # Subsample model projections to match md projections size
    min_sample_size = min(len(md_projections), len(re_projections), len(model_projections), len(mdft_projections))
    np.random.seed(42)  # Set seed for reproducibility
    model_projections = model_projections[np.random.choice(model_projections.shape[0], size=min_sample_size, replace=False)]
    re_projections = re_projections[np.random.choice(re_projections.shape[0], size=min_sample_size, replace=False)]
    md_projections = md_projections[np.random.choice(md_projections.shape[0], size=min_sample_size, replace=False)]
    mdft_projections = mdft_projections[np.random.choice(mdft_projections.shape[0], size=min_sample_size, replace=False)]

    # Calculate histograms from re projections
    counts_md, xedges, yedges = np.histogram2d(md_projections[:,0], md_projections[:,1], bins=100)
    counts_md = counts_md.T  #  Transpose to match the orientation of pcolormesh
    counts_re, xedges, yedges = np.histogram2d(re_projections[:,0], re_projections[:,1], bins=100)
    counts_re = counts_re.T 
    counts_model, _, _ = np.histogram2d(model_projections[:,0], model_projections[:,1], bins=100)
    counts_model = counts_model.T  # Transpose to match the orientation of pcolormesh
    counts_mdft, _, _ = np.histogram2d(mdft_projections[:,0], mdft_projections[:,1], bins=100)
    counts_mdft = counts_mdft.T  # Transpose to match the orientation of pcolormesh
    edges.append((xedges, yedges))
    # Normalize histograms
    pmf_md = counts_md / np.sum(counts_md)
    pmf_re = counts_re / np.sum(counts_re)
    pmf_model = counts_model / np.sum(counts_model)
    pmf_mdft = counts_mdft / np.sum(counts_mdft)

    # Calculate negative log of probability (free energy)
    with np.errstate(divide='ignore'):
        fe_md = -np.log(pmf_md)
        fe_re = -np.log(pmf_re)
        fe_model = -np.log(pmf_model)  
        fe_mdft = -np.log(pmf_mdft) 
    # Replace infinities with NaN
    fe_md[~np.isfinite(fe_md)] = np.nan
    fe_model[~np.isfinite(fe_model)] = np.nan
    fe_mdft[~np.isfinite(fe_mdft)] = np.nan
    md.append(fe_md)
    re.append(fe_re)
    tito.append(fe_model)
    mdft.append(fe_mdft)


In [ ]:
n_blocks = 4

# -----------------------------
# Layout setup
# -----------------------------
fig = plt.figure(figsize=(16, 16))
outer = gridspec.GridSpec(n_blocks, 1, hspace=0.001) #, wspace=0.35

# Loop over blocks
for block_idx in range(n_blocks):
    # --- Left column (MD density)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 5,  # extra column for colorbar
        subplot_spec=outer[block_idx],
        hspace=0.1,
        wspace=0.3,
        width_ratios=[ 1, 1, 1, 1, 0.25]  # spacer before colorbar, then ax3
    )
    ax1 = fig.add_subplot(inner[0,0])
    #ax1 = fig.add_subplot(outer[block_idx, 0])
    
    if block_idx == 0:
        ax1.set_title("MD", fontproperties=font_prop)

    ax1.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax1.set_aspect('equal')
    ax1.set_box_aspect(1)
    # Apply font properties to tick labels
    ax1.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    # Set 5 ticks for both axes
    ax1.locator_params(axis='both', nbins=5)

    # Plot MD density
    pcm = ax1.pcolormesh(edges[block_idx][0], edges[block_idx][1], md[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto')
    vmin, vmax = pcm.get_clim()
    

    ax2 = fig.add_subplot(inner[0, 1])
    if block_idx == 0:
        ax2.set_title("RE", fontproperties=font_prop)
    if block_idx == n_blocks - 1:
        ax2.set_xlabel('TIC 1', fontproperties=font_prop)
    #ax2.set_ylabel('TIC 2', fontproperties=font_prop)
    #ax2.set_aspect('equal')
    ax2.set_box_aspect(1)
    # Apply font properties to tick labels
    ax2.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax2.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax2.pcolormesh(edges[block_idx][0], edges[block_idx][1], re[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD


    # --- Middle right column (Timescale scatter)
    ax3 = fig.add_subplot(inner[0, 2])
    if block_idx == 0:
        ax3.set_title("TITO+MDFT", fontproperties=font_prop)
    ax3.set_box_aspect(1)
    # Apply font properties to tick labels
    ax3.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax3.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax3.pcolormesh(edges[block_idx][0], edges[block_idx][1], mdft[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # --- Right column (Timescale scatter)
    ax4 = fig.add_subplot(inner[0, 3])
    if block_idx == 0:
        ax4.set_title("TITO", fontproperties=font_prop)
    ax4.set_box_aspect(1)
    # Apply font properties to tick labels
    ax4.tick_params(axis='both', which='major', labelsize=font_prop.get_size())
    
    
    # Set 5 ticks for both axes
    ax4.locator_params(axis='both', nbins=5)

    # Plot Model density
    pcm = ax4.pcolormesh(edges[block_idx][0], edges[block_idx][1], tito[block_idx],
                         cmap="viridis_r",   # Use reversed viridis for high density as bright
                         shading='auto',
                         vmin=vmin, vmax=vmax)  # Use same color scale as MD
    
    # Create a separate axes for the colorbar to avoid affecting the main plot size
    cbar_ax = fig.add_axes([ax4.get_position().x1 + 0.01,
                            ax4.get_position().y0,
                            0.01,
                            ax4.get_position().height])
    cbar = fig.colorbar(pcm, cax=cbar_ax)
    cbar.set_label("Free energy $[k_BT]$", fontproperties=font_prop, labelpad=5)
    # Apply font properties to tick labels
    cbar.ax.tick_params(labelsize=font_prop.get_size())

plt.tight_layout(rect=[0.05, 0, 1, 1])  # leave more space on the left for labels
plt.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/appendix/examples_tito_more_re.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/appendix/examples_tito_more_re.pdf", transparent=True)

## Compute calibration

In [ ]:
data_set = "timewarp"
sub_data_set = "large"  # "large", "huge"
model = "lxf356pw" # uniform: 57hk8qzx, lxf356pw. exp: ktgl41l5, dg5fl7wv
split = "test"
mol_index = 1 #1 , 84
lag = 250  #250 125
nested_samples = 500  #500 400
ode_steps_list = [20, 30, 40, 50]  #20, 40

md_ts = []
model_ts = []
vamp2_gap = []

for ode_steps in ode_steps_list:
    #load singular values

    md_svs = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/md/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values.npy")
    #vamp_gap = np.load()
    model_svs = np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_singular_values_random_init_lag_{int(lag)}_nested_{nested_samples}_ode_steps_{ode_steps}.npy") #short_params. _ode_steps_{ode_steps}
    vamp2_gap.append(np.load(f"{RESULTS_DIR}/{data_set}/{sub_data_set}/{model}/{split}/mol_{str(mol_index).zfill(5)}/vamp_gap_random_init_lag_{int(lag)}_nested_{nested_samples}_ode_steps_{ode_steps}.npy")) ##short_params. _ode_steps_{ode_steps}
    #compute implied timescales
    md_ts.append(-lag/np.log(md_svs))
    ts_indices = np.where(md_ts[-1] > lag/5)
    md_ts[-1] = md_ts[-1][ts_indices]
    model_ts.append(-lag/np.log(model_svs))
    model_ts[-1] = model_ts[-1][ts_indices]

In [ ]:
vamp2_gap

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

# Plot timescale correlations for each ode_steps value
for i, ode_steps in enumerate(ode_steps_list):
    ax = axes[i // 2, i % 2]
    
    # Scatter plot
    ax.scatter(md_ts[i], model_ts[i], color=palette_colors[1], s=200, alpha=0.7)
    
    # Set log scale
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    # Equal limits
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    low = min(xmin, ymin)
    high = max(xmax, ymax)
    
    # Diagonal line (y=x)
    ax.plot([low, high], [low, high], color=palette_colors[0], 
            linestyle='dashed', linewidth=2, label='y=x')
    
    ax.plot([lag for _ in range(10)], np.arange(low, lag, (lag-low)/10), color=palette_colors[-1], linestyle='dotted',
     linewidth=2, markersize=50, zorder=1)
    ax.plot(np.arange(low, lag, (lag-low)/10), [lag for _ in range(10)], color=palette_colors[-1], linestyle='dotted',
     linewidth=2, markersize=50, zorder=1)
    
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    
    # Labels and title
    if i == 2 or i == 3:
        ax.set_xlabel('Timescales MD [ps]', fontproperties=font_prop)
    if i == 0 or i == 2:
        ax.set_ylabel('Timescales TITO [ps]', fontproperties=font_prop)

    ax.set_title(f'ODE Steps: {ode_steps}', fontproperties=font_prop)
    
    # Calculate and display relative error
    rel_error = np.mean(np.abs((model_ts[i][:10] - md_ts[i][:10]) / md_ts[i][:10]))
    ax.text(0.05, 0.9, f'Relative error = {rel_error:.3f}', transform=ax.transAxes, fontproperties=font_prop)
    ax.text(0.05, 0.8, f'VAMP-2 gap  = {vamp2_gap[i]:.3f}', transform=ax.transAxes, fontproperties=font_prop)
    #this two we don't to report here for the final figures, just aggregated
    ax.tick_params(axis='both', which='major', labelsize=font["size"])

plt.tight_layout()
plt.show()

# #Relative error calculation
# rel_error = np.mean(np.abs((model_ts[:10] - md_ts[:10]) / md_ts[:10]))

# fig, axes = plt.subplots(figsize=(6, 6))
# axes.scatter(md_ts, model_ts, color=palette_colors[1], s=300, zorder=2)
# #axes.plot(md_ts, fit_line, color=palette_colors[2], linestyle='solid', linewidth=2, markersize=12, zorder=1)

# #Set log scale
# axes.set_xscale('log')
# axes.set_yscale('log')
# plt.axis('square')
# axes.set_box_aspect(1)

# #Set equal range in both axis
# # Get current limits
# xmin, xmax = axes.get_xlim()
# ymin, ymax = axes.get_ylim()

# # Find the combined min and max
# low = min(xmin, ymin)
# high = max(xmax, ymax)

# axes.plot(np.arange(low, high, (high-low)/10), np.arange(low, high, (high-low)/10), color=palette_colors[0], linestyle='dashed',
#      linewidth=2, markersize=12, zorder=1, label='y=x')

# # Apply to both axes
# axes.set_xlim(low, high)
# axes.set_ylim(low, high)

# axes.plot([lag for _ in range(10)], np.arange(low, lag, (lag-low)/10), color=palette_colors[-1], linestyle='dotted',
#      linewidth=2, markersize=12, zorder=1)
# axes.plot(np.arange(low, lag, (lag-low)/10), [lag for _ in range(10)], color=palette_colors[-1], linestyle='dotted',
#      linewidth=2, markersize=12, zorder=1)

# axes.text(0.05, 0.9, f'Relative error = {rel_error:.3f}', transform=axes.transAxes, fontproperties=font_prop)
# axes.text(0.05, 0.8, f'VAMP-2 gap  = {vamp2_gap:.3f}', transform=axes.transAxes, fontproperties=font_prop)
# #this two we don't to report here for the final figures, just aggregated
# plt.tick_params(axis='both', which='major', labelsize=font["size"])

# axes.set_xlabel('Timescale MD [$ps$]', fontproperties=font_prop)
# axes.set_ylabel('Timescale model [$ps$]', fontproperties=font_prop)

# fig.tight_layout()
# fig.show()

In [ ]:
import os
os.makedirs(osp.dirname(f"../plots/paper_figures/appendix/calibration.pdf"), exist_ok=True)
fig.savefig(f"../plots/paper_figures/appendix/calibration.pdf", transparent=True)